# Plan the Deployment

The shares of points sampled so far are those of each design's full constellation, as if all of its satellites flew for the whole mission. In fact, they are launched over months or years, and fewer satellites sample the points meanwhile. This notebook accounts for the deployment of each design, then asks whether to commit to its whole constellation at once.

It covers:

* the **phases** of a deployment, and what a design achieves over its mission;
* **staged** deployments, whose later stages are deployed only if a rule decides to.

It builds on `TradeCostAndUtility`, whose mission plan dates the launches of each design, and on `CompareScenarios`, whose fields the rules of the staged deployments respond to.

## Account for Deployment

The launches of a design divide its mission into phases, in each of which some of its satellites fly. With `phases`, the study evaluates each phase with the observations of its satellites (over the same day, from the cache) and combines the phases over the mission:

* `sufficient mission`: the share of points sufficiently sampled, on average over the five years from the first launch;
* `sufficient time`: the years from the start of development until a requirement is met, here that half of the points are sampled sufficiently.

The phased study evaluates the efficient designs of `EvaluateTradeSpace`: the most sufficient design of each number of satellites and planes.

In [1]:
from trade_space_example import OBJECTIVES, cost_study, describe
from tatc.tradespace import Requirement, aggregate, pareto_front, summarize

summary = summarize(cost_study.run(), ["satellites", "planes", "sufficient", "accurate"])
summary["description"] = summary.apply(describe, axis=1)
front = summary[pareto_front(summary, OBJECTIVES)]
leaders = front.sort_values("sufficient", ascending=False).drop_duplicates(["satellites", "planes"])

phased_study = cost_study.replace(phases=[Requirement("sufficient", ">=", 50)])
phased = phased_study.run(leaders)
PHASED = ["deployed", "sufficient", "sufficient mission", "sufficient time"]
deployed = aggregate(phased, PHASED, design_columns=[]).set_index("design")[PHASED]
leaders = leaders.set_index("design")[["description", "satellites", "planes"]].join(deployed)
leaders["lost"] = leaders["sufficient"] - leaders["sufficient mission"]
leaders.sort_values(["satellites", "planes"]).round(1)

,description,satellites,planes,deployed,sufficient,sufficient mission,sufficient time,lost
design,,,,,,,,
163120bf9590,"800 km, 60 deg FOR, SSO: 1 at 18:00",1.0,1.0,3.2,3.2,3.2,inf,0.0
f7413b1f8466,"800 km, 60 deg FOR, Walker 45 deg: 1 x 2 (spac...",2.0,1.0,3.5,16.6,16.6,inf,0.0
33b006848799,"800 km, 60 deg FOR, Walker 45 deg: 2 x 1 (spac...",2.0,2.0,3.8,30.2,27.3,inf,2.9
d5d3fa915ca7,"800 km, 60 deg FOR, Walker 45 deg: 1 x 3 (spac...",3.0,1.0,3.8,38.0,38.0,inf,0.0
1af5f2707be3,"800 km, 60 deg FOR, Walker 45 deg: 1 x 4 (spac...",4.0,1.0,4.5,68.2,63.5,4.5,4.7
6ba50f2632cd,"800 km, 60 deg FOR, Walker 45 deg: 2 x 2 (spac...",4.0,2.0,4.0,69.6,64.3,4.0,5.3
6bef11e40977,"800 km, 60 deg FOR, Walker 75 deg: 2 x 3 (spac...",6.0,2.0,4.5,97.8,87.1,4.5,10.7


Deployment costs the designs of several planes or launches. Six satellites in two planes sample 98% of the points once deployed, but 87% on average over the mission: the first plane flies alone for three quarters of a year while the second is produced, and half of the points are sampled sufficiently only 4.5 years after development starts. Four satellites lose about 5 points of their 68 or 70%, in one plane (which takes two launches of up to three satellites) or in two, and two satellites in two planes lose 3 points of their 30%. The designs launched at once lose nothing, and the designs of fewer than four satellites never sample half of the points (an infinite `sufficient time`). A shorter mission, slower production, or rarer launches would weigh deployment more, and could reorder the front.

## Stage the Deployment

A deployment need not commit to every plane at once. A `Stage` is a set of planes that is produced and launched only if a rule decides to, at given times of the mission. The rule decides from what is known then (a `Decision`) and from the values of the design. A stage that is not deployed is not produced, launched, or paid for. Staging saves the cost of a stage where the mission does not need it, and loses what the stage would have achieved while it waits, where the mission does.

Here, the robust design of four satellites of `CompareScenarios` (two sun-synchronous planes of two) stages its second plane in one of five ways, a new decision of the design (`staging`):

| staging | the second plane is deployed |
|---|---|
| at once | with the first |
| first plane | never |
| decided at launch | if the field decorrelates within 6 hours, decided at the first launch |
| decided a year later | if the field decorrelates within 6 hours, decided a year after the first launch |
| measured at launch | if the first plane samples fewer than half of the points accurately, measured at its launch (`Decision.capability`) |

The rules that read the field read the scenario, which a mission would not know; the last rule reads only what the first plane achieves. The `stages` of the plan are a function of the design. Each staging is evaluated in each field of `CompareScenarios`, for the share of points accurately sampled on average over the mission, and the present value of the cost (`cost present`, discounted at 3% a year):

In [2]:
import dataclasses
from datetime import timedelta

from trade_space_example import DECORRELATION_TIME, FIELDS, PLAN, cost_space, sample_field
from tatc.tradespace import Categorical, Fixed, Stage

STAGING = ["at once", "first plane", "decided at launch", "decided a year later", "measured at launch"]
staging_space = cost_space.extend(decorrelation_time=Fixed(DECORRELATION_TIME), staging=Categorical(STAGING))


def stages(d):
    """The stages of a design's deployment: its second plane, by its staging."""
    if d.staging == "at once":
        return []
    if d.staging == "first plane":
        return [Stage(1, 0.0, lambda decision: False)]
    if d.staging == "measured at launch":
        return [Stage(1, 0.0, lambda decision: decision.capability["accurate"] < 50)]
    when = 0.0 if d.staging == "decided at launch" else 1.0  # years after the first launch
    return [Stage(1, when, lambda decision: d.decorrelation_time < timedelta(hours=6))]


staged_study = phased_study.replace(
    space=staging_space, metrics=sample_field, plan=dataclasses.replace(PLAN, stages=stages), scenarios=FIELDS
)
(four,) = cost_space.designs_of(summary[summary.description == "800 km, 60 deg FOR, SSO: 2 at 15:00, 2 at 09:00"])
staged = staged_study.run([staging_space.design(four, staging=option) for option in STAGING])
staged.groupby(["staging", "scenario"], sort=False)[["accurate mission", "cost present"]].mean().unstack(
    "scenario"
).round(1)

accurate mission                  cost present          \
scenario                       1 hour 3 hours 12 hours       1 hour 3 hours   
staging                                                                       
at once                          62.6    77.9     91.0        156.4   156.4   
first plane                       4.8    10.0     73.0        118.8   118.8   
decided at launch                62.6    77.9     73.0        156.4   156.4   
decided a year later             49.7    62.8     73.0        154.5   154.5   
measured at launch               62.6    77.9     73.0        156.4   156.4   

                               
scenario             12 hours  
staging                        
at once                 156.4  
first plane             118.8  
decided at launch       118.8  
decided a year later    118.8  
measured at launch      118.8

Deciding at the first launch is worth the most. Where the field varies within hours, the second plane is produced and launched as if it were deployed at once: 63% and 78% of the points are accurately sampled on average over the mission, for 156 million dollars. Where the field varies slowly, the second plane is not deployed, which saves 38 million dollars (24%) for 18 points of the mean (73% against 91%): a trade that the rule makes, and that a mission might not.

Deciding a year later costs the fields that need the second plane 13 to 15 points of the mean (50% and 63%), as the first plane flies alone a year longer, and saves only 2 million dollars, by discounting the later production. The earlier the need is known, the more staging saves, at less loss.

The rule that measures makes the same choices as the rule that reads the field, in every field. The first plane alone samples 5% and 10% of the points accurately where the field varies within hours, so the second plane is deployed, and 73% where it varies over 12 hours, so it is not. A mission knows what its satellites achieve, not the field, so this is a rule it could follow.

## Summary

* A `DeploymentSchedule` divides each design's mission into phases of the satellites that fly, which the study evaluates (with `phases`) for what the design achieves on average over its mission, and the time until it meets a requirement.
* A `Stage` of a deployment is produced and launched only if a rule decides to, from what is known then (a `Decision`, such as what the satellites in orbit achieve, `Decision.capability`) and the values of the design. A study measures what the option is worth.

The next notebook, `PlanForFailures`, simulates the failures of satellites and launches over each mission.

## Notes

* Each phase is evaluated over the same day of July, as if the geometry of the satellites and the season were those of that day throughout the mission (quasi-static phases, see `evaluate_phases`). This misses the drift between planes of different inclinations or altitudes, and seasons. The planes of these designs share an altitude and an inclination, and the shares do not depend on the sun, so the day should represent the mission. An `Uncertain` start of the observations, drawn for each replicate over a year, would average over the drift and the seasons, at the cost of analyzing the satellites again for each day.
* The rule that measures the first plane reads its capability at once, as the study computes it. A real measurement takes time, and a rule that waits a year to measure loses what deciding a year later loses.
* A rule may also read the satellites still operating (see `Decision`), for a reserve plane launched only if satellites fail.